Vamos a ver cómo se puede entrenar un modelo para clasificar diferentes acciones típicas de una tienda.

¿Qué es una librería?

Una librería es un conjunto de funciones ya desarrolladas que podemos reutilizar en nuestros programas.

En lugar de programarlo todo desde cero, utilizamos herramientas creadas por otros desarrolladores.

Antes de empezar hay que descargar la librería Sentence Transformers que convierte el texto a vectores.

In [46]:
%pip install sentence-transformers

Note: you may need to restart the kernel to use updated packages.


Y la librería Scikit-Learn que contiene diferentes modelos de ML.

In [47]:
%pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [48]:
from sentence_transformers import SentenceTransformer

In [49]:
from sklearn.linear_model import LogisticRegression

# Ejercicio 1 — Crear nuevas categorías

Hasta ahora hemos trabajado con tres intenciones:

- compra
- devolucion
- saludo

Vamos a ampliar nuestro clasificador.

## Tarea

Añade una nueva categoría llamada:

**problema**

Crea al menos 4 ejemplos de frases que pertenezcan a esta categoría.

Después:

1. Añade los ejemplos al conjunto de entrenamiento.
2. Genera de nuevo los embeddings.
3. Entrena de nuevo el clasificador.
4. Prueba el modelo con al menos 5 frases nuevas.

## Preguntas

- ¿Todas las frases se clasifican correctamente?
    
    Al principio no, he tenido que modificar la regularizacion de para que el modelo no fuera restringido y volver a entrenarlo mas ajustado a los datos, una vez hecho eso, se han clasificado todas de manera correcta
- ¿Hay alguna predicción que te sorprenda?
    
    La respuesta de "aupa", ya que se usa como saludo y lo ha sabido relacionar. Me imagino que al ser solo una palabra vera la similitud con "hola, donde se ha dicho que era saludo y solo era una palabra.
- ¿Qué ocurre si escribes una frase ambigua?

    Cuando he escrito una frase ambigua,  ha seleccionado como categoria, problema, me imagino que al haber el doble de registros que en el resto de categorias por probabilidad escoge problema

1. Datos.

In [50]:
train_data = [
    ("quiero comprar un ordenador", "compra"),
    ("necesito un portátil", "compra"),
    ("quiero devolver mi pedido", "devolucion"),
    ("mi producto ha llegado roto", "devolucion"),
    ("buenos días", "saludo"),
    ("hola", "saludo"),
    ("esta roto el ordenador", "problema"),
    ("mi portátil no funciona", "problema"),
    ("no puedo acceder a mi cuenta", "problema"),
    ("se me ha caido el ordenador", "problema"),
    ("me apetece un helado", "compra"),
    ("no me gusta este producto", "devolucion"),
    ("quiero cambiar mi pedido", "devolucion"),
    ("kaixo", "saludo"),
    ("egun on", "saludo"),
    ("tengo una nueva camiseta", "compra"),
    ("necesito atencion con mi pedido", "problema"),
    ("ayuda", "problema"),
    ("el paquete ha llegado defectuoso", "devolucion"),
    ("quiero un reembolso", "devolucion")

]

In [51]:
train_data_mejorado = [
    # Categoría: compra
    ("quiero comprar un ordenador", "compra"),
    ("necesito un portátil", "compra"),
    ("me apetece un helado", "compra"),
    ("tengo una nueva camiseta", "compra"),
    ("quiero adquirir un teléfono móvil", "compra"),
    ("estoy buscando unos auriculares inalámbricos", "compra"),
    ("¿cuánto cuesta este ordenador?", "compra"),
    ("me gustaría comprar una televisión nueva", "compra"),
    ("necesito comprar un regalo para mi hermano", "compra"),
    ("quiero hacer un pedido de varios productos", "compra"),
    ("¿tenéis portátiles disponibles?", "compra"),
    ("me interesa adquirir este producto", "compra"),
    ("quisiera comprar una mochila para la universidad", "compra"),
    ("voy a comprar un teclado para mi ordenador", "compra"),
    ("necesito renovar mi teléfono y quiero ver los modelos disponibles", "compra"),

    # Categoría: devolucion
    ("quiero devolver mi pedido", "devolucion"),
    ("mi producto ha llegado roto", "devolucion"),
    ("no me gusta este producto", "devolucion"),
    ("quiero cambiar mi pedido", "devolucion"),
    ("el paquete ha llegado defectuoso", "devolucion"),
    ("quiero un reembolso", "devolucion"),
    ("me gustaría devolver los zapatos que compré", "devolucion"),
    ("¿cómo puedo solicitar una devolución?", "devolucion"),
    ("he recibido un producto equivocado y quiero cambiarlo", "devolucion"),
    ("quiero recuperar el dinero de mi compra", "devolucion"),
    ("el artículo no es como aparecía en las fotos", "devolucion"),
    ("necesito cambiar la talla de la camiseta", "devolucion"),
    ("me arrepiento de la compra y quiero devolver el producto", "devolucion"),
    ("el pedido que he recibido no corresponde con lo que solicité", "devolucion"),
    ("quisiera saber cuánto tiempo tengo para devolver un artículo y recibir el reembolso", "devolucion"),

    # Categoría: saludo
    ("buenos días", "saludo"),
    ("hola", "saludo"),
    ("kaixo", "saludo"),
    ("egun on", "saludo"),
    ("buenas tardes", "saludo"),
    ("buenas noches", "saludo"),
    ("hola, ¿qué tal?", "saludo"),
    ("muy buenas", "saludo"),
    ("buenos días, ¿hay alguien disponible?", "saludo"),
    ("hola, me gustaría hacer una consulta", "saludo"),
    ("kaixo, zer moduz?", "saludo"),
    ("buenas, espero que estéis bien", "saludo"),
    ("hola, necesito hablar con atención al cliente", "saludo"),
    ("egun on, tengo una pregunta sobre un pedido", "saludo"),
    ("buenos días, me pongo en contacto con vosotros para solicitar información", "saludo"),

    # Categoría: problema
    ("esta roto el ordenador", "problema"),
    ("mi portátil no funciona", "problema"),
    ("no puedo acceder a mi cuenta", "problema"),
    ("se me ha caido el ordenador", "problema"),
    ("necesito atencion con mi pedido", "problema"),
    ("ayuda", "problema"),
    ("mi ordenador no enciende", "problema"),
    ("la pantalla del móvil está bloqueada", "problema"),
    ("no puedo iniciar sesión en la página web", "problema"),
    ("mi pedido todavía no ha llegado", "problema"),
    ("la aplicación da error cada vez que intento abrirla", "problema"),
    ("he olvidado mi contraseña y no consigo recuperar el acceso", "problema"),
    ("el pago se ha realizado, pero no he recibido ninguna confirmación", "problema"),
    ("mi ordenador funciona muy lento y se apaga inesperadamente", "problema"),
    ("tengo un problema con mi pedido porque el seguimiento no se actualiza desde hace varios días", "problema"),
]

2. Modelo preentrenado.

In [52]:
model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

embedding = model.encode(
    "quiero devolver mi pedido"
)

print(embedding.shape)

print(embedding[:10])

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5572.64it/s]


(384,)
[-0.01777516  0.06185898  0.00256953 -0.03221566 -0.07757559 -0.06224593
  0.09503879  0.05418092  0.05467332  0.06182623]


3. Entrenamos el clasificador.

In [53]:
texts = [x[0] for x in train_data]
labels = [x[1] for x in train_data]

X = model.encode(texts)

In [54]:
# Entrenar el modelo
#clf = LogisticRegression()
# C10 permite al modelo ajustarse mejor a los datos de entrenamiento
# max_iter=1000 permite al modelo converger en más iteraciones si es necesario
clf = LogisticRegression(C=10, max_iter=1000) 

clf.fit(X, labels)

,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",10
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in

4. Hacemos la predicción.

In [55]:
nueva_frase = "quiero cambiar mi producto"
embedding = model.encode(
    [nueva_frase]
)

pred = clf.predict(embedding)

print(pred)

['devolucion']


In [56]:
frase_problema = "mi portátil no funciona"
embedding = model.encode(
    [frase_problema]
)

pred = clf.predict(embedding)

print(pred)

['problema']


In [57]:
frase_compra = "quiero comprar un ordenador"
embedding = model.encode(
    [frase_compra]
)

pred = clf.predict(embedding)

print(pred)

['compra']


In [58]:
frase_ambigua = "los minions son muy graciosos"
embedding = model.encode(
    [frase_ambigua]
)

pred = clf.predict(embedding)

print(pred)

['devolucion']


In [59]:
frase_saludo = "aupa"
embedding = model.encode(
    [frase_saludo]
)
pred = clf.predict(embedding)
print(pred)

['saludo']


## Ejercicio 2 — Probar frases nuevas y ambiguas

El modelo no debe probarse solamente con frases muy parecidas a las utilizadas durante el entrenamiento. Ahora debes comprobar cómo se comporta con formas diferentes de expresar la misma intención.

Tareas:

1. Crea al menos 10 frases nuevas que NO aparezcan en train_data.

2. Incluye frases relacionadas con compra, devolución, saludo y problema.

3. Escribe al menos 3 frases ambiguas, que puedan pertenecer razonablemente a más de una categoría.

4. Obtén la predicción del modelo para todas ellas.

5. Crea una tabla con: frase, categoría esperada, categoría predicha y si ha acertado.

# Preguntas:

¿Funciona igual de bien cuando cambias completamente la forma de escribir una intención?

No porque el modelo no ha sido entrenado con esa forma y es mas probable que se equivoque

¿Qué tipos de frases provocan más errores?

Aquellas que contienen palabras o patrones con los que el modelo no se ha entrenado

¿Qué ocurre con una frase que puede interpretarse como dos categorías diferentes?

Que es mas probable que selecciones, sobre aquella categoria que mas se repite por probabilidad

¿El modelo entiende realmente la intención o está utilizando similitudes presentes en los ejemplos?

Esta utilizando similitudes similares, por ejemplo en el saludo se repite el patron de pocas palabras ("hola", "egun on") entonces
al escribir otro saludo cualquiera al ser la categoria que mas se repite con pocas palabras lo mas probable es que lo una a la categoria de saludo


Entrega: Tabla de al menos 10 frases y una conclusión breve sobre los errores observados.


In [60]:
frase_dudosa_compra = "toma esta camiseta quiero otra"
embedding = model.encode(
    [frase_dudosa_compra]
)
pred = clf.predict(embedding)
print(pred)

['devolucion']


In [61]:
frase_dudosa_saludo = "hellow, how are you?"# La lee como saludo
frase_dudosa_saludo = "aupa, que tal?"
embedding = model.encode(
    [frase_dudosa_saludo]
)
pred = clf.predict(embedding)
print(pred)

['compra']


In [62]:
frase_dudosa_problema = "esto deberia enciender y el tuyo ?"
embedding = model.encode(
    [frase_dudosa_problema]
)
pred = clf.predict(embedding)
print(pred)

['problema']


In [63]:
frase_dudosa_devolucion= "quiero una talla menos de camiseta"
embedding = model.encode(
    [frase_dudosa_devolucion]
)
pred = clf.predict(embedding)
print(pred)
print (embedding)
embedding.shape

['devolucion']
[[ 8.01189430e-03  1.05804861e-01 -4.55554649e-02 -6.69945478e-02
  -2.51473170e-02 -1.08162696e-02  9.62237790e-02  9.60954279e-02
  -2.33035181e-02  9.24391896e-02  5.26212156e-02 -5.50215766e-02
  -5.72812296e-02  3.03769819e-02  3.34177874e-02 -7.21683279e-02
  -4.05917838e-02  2.70209480e-02  1.83893349e-02  7.37058967e-02
   2.59701591e-02 -5.30686714e-02 -7.41504729e-02  9.52772424e-02
  -9.60216448e-02 -4.82050367e-02 -2.70879045e-02  6.81417286e-02
  -2.84173917e-02 -5.01660258e-02 -5.87111302e-02  6.36835769e-02
   9.34410766e-02 -2.91905887e-02 -4.34862915e-03 -2.30673440e-02
   6.28866702e-02 -9.59570631e-02 -3.17491628e-02  1.02514870e-01
  -1.16625674e-01 -1.51514737e-02 -4.56651486e-02  1.25559457e-02
   4.41713966e-02 -3.85170081e-03  6.88948184e-02  7.45135769e-02
   2.25194357e-03 -8.10446292e-02 -8.06476623e-02 -3.97147313e-02
  -2.92887744e-02  4.63419184e-02 -7.88754411e-03  1.86906233e-02
  -4.11560498e-02 -6.22519702e-02  6.54312819e-02  4.72571924

(1, 384)

In [64]:
frase_dudosa_compra= "quiero pedir mas camisetas al Olentzero"
embedding = model.encode(
    [frase_dudosa_compra]
)
pred = clf.predict(embedding)
print(pred)

['devolucion']


In [65]:
frase_dudosa_problema= "se me ha roto el ordenador 5 años despues"
embedding = model.encode(
    [frase_dudosa_problema]
)
pred = clf.predict(embedding)
print(pred)
print (embedding)
embedding.shape

['problema']
[[-3.93888243e-02  2.12117992e-02  4.86572385e-02 -2.97098793e-02
   1.78603083e-02 -4.17333134e-02  5.58892414e-02  4.59684152e-03
   6.28773421e-02  6.29365295e-02  7.30500594e-02 -3.19025777e-02
   1.57768913e-02 -4.33597043e-02 -4.85576764e-02  2.95832232e-02
  -1.77120287e-02  7.86327794e-02 -9.07838196e-02  1.99212367e-03
   6.88346252e-02  1.66380703e-02 -1.33304432e-01  8.99142772e-02
  -1.10980824e-01  7.81193329e-03 -2.38560271e-02  1.48507664e-02
  -4.99075390e-02 -1.23652183e-01 -5.15041277e-02  1.90964490e-02
   7.28815794e-02  2.93730460e-02 -6.07938915e-02  2.02256674e-03
   6.20668121e-02 -8.02925453e-02 -6.10065311e-02  4.64914180e-02
  -6.99079558e-02  1.08474456e-02 -2.61404645e-02 -1.05031691e-02
   4.39610742e-02 -6.22576661e-02 -3.19065228e-02  2.24397294e-02
   6.42214268e-02 -2.79012993e-02 -2.01746095e-02  2.87292786e-02
   2.19115103e-03  1.68195814e-02 -7.19874576e-02  1.60182416e-02
   2.26415582e-02 -4.12965678e-02  9.05224010e-02 -5.73261641e-

(1, 384)

In [66]:
frase_dudosa_saludo= "eguerdi on, dena ondo gazteok?"
embedding = model.encode(
    [frase_dudosa_saludo]
)
pred = clf.predict(embedding)
print(pred)

['devolucion']


In [67]:
#frase_dudosa_contexto= "mi equipo favorito es el Athletic de Bilbao"
frase_dudosa_contexto= "El Athletic de Bilbao"
#frase_dudosa_contexto= "El Athletic"
embedding = model.encode(
    [frase_dudosa_contexto]
)
pred = clf.predict(embedding)
print(pred)
print(embedding)
embedding.shape

['devolucion']
[[ 3.44456770e-02  2.98677832e-02 -4.26574647e-02  1.17467772e-02
   1.83885191e-02  5.41715883e-02  7.28177698e-03 -8.59874766e-04
   8.81336629e-02  1.00386525e-02  1.06613347e-02 -6.19308539e-02
   3.84700447e-02  2.57716291e-02  6.29368573e-02 -5.76365134e-03
  -1.90358441e-02 -4.24688570e-02  1.33055132e-02 -1.95652861e-02
   6.13232739e-02 -2.96380091e-02 -5.16312197e-02  2.76052840e-02
  -6.43758029e-02 -2.55452842e-02  1.09274779e-02 -2.00681924e-03
  -8.22299197e-02 -6.28667846e-02  1.06729073e-02 -6.14191182e-02
   5.49026616e-02  6.27971068e-03  1.19406544e-02 -9.47301835e-03
  -9.14611109e-03 -4.15828712e-02 -5.48826996e-03  6.06772862e-02
  -1.10127576e-01 -4.68715988e-02 -4.60732840e-02  2.22746935e-02
   3.97638939e-02 -3.15690972e-02  2.02953666e-02  4.84594107e-02
  -2.18846183e-02  5.77985868e-02  3.76539379e-02 -3.99501696e-02
   1.01758819e-02 -8.75363350e-02 -1.11206230e-02  3.89948376e-02
  -3.28656882e-02  2.62012277e-02  1.09896861e-01  5.20963036

(1, 384)

Tabla resumen:

| Frase | Categoria esperada | Categoria predicha |¿Correcta? |
|---|---|---|---|
| "toma esta camiseta quiero otra" | Devolución | Devolución | Sí |
| "hellow, how are you?" | Saludo | Saludo | Sí |
| "aupa, que tal?" | Saludo |  Compra | No|
| "esto deberia enciender y el tuyo ?" | Problema | Problema | Sí |
| "quiero una talla menos de camiseta" | Devolucion | Devolucion | Sí |
| "quiero pedir mas camisetas al Olentzero" | Compra | Devolucion | No |
| "se me ha roto el ordenador 5 años despues" | Problema | Problema | Sí |
| "eguerdi on, dena ondo gazteok?" | Saludo | Devolucion | No |
| "mi equipo favorito es el Athletic de Bilbao" | Sin categoria esperada | Problema | No |
| "El Athletic de Bilbao" | Sin categoria esperada | Saludo | No |
| "El Athletic" | Sin categoria esperada | Devolucion | No |

En este caso por el modelo entregado, y en los errores adquiridos han sido por seguido el patron donde las frases mas cortas son las de saludo las intermedias las de devolucion y las mas largas las de problemas.


## Ejercicio 3 — Investigar qué es un embedding
En el notebook, SentenceTransformer convierte cada frase en un vector numérico. El objetivo es investigar qué significa esta transformación.

Tareas:

6. Muestra el embedding de al menos 3 frases diferentes.

7. Comprueba su tamaño utilizando. shape.

8. Compara dos frases que expresen una intención parecida y otras dos que expresen intenciones diferentes.

9. Investiga en la documentación de Sentence Transformers cómo se puede calcular la similitud entre embeddings.

10. Utiliza esa información para comprobar qué frases son más parecidas entre sí.

# Preguntas:

¿Cuántos valores tiene el embedding generado por el modelo utilizado?

En este caso tiene 384 valores

¿Por qué un texto puede representarse mediante una lista de números?

Por que la informacion se transorma a una lista de numeros, la cual crea vectores

¿Qué significa que dos frases tengan embeddings similares?

Que son dos frases similares

¿La similitud entre embeddings es exactamente lo mismo que pertenecer a la misma categoría?

No, la similitud entre embeddings quiere decir que pueden tener un significados de palabras parecido, pero debido al contexto pueden ser de diferentes categorias

Entrega: Código utilizado y una pequeña tabla con pares de frases y sus similitudes.

In [68]:
frase_correcta_devolucion = "quiero modificar mi pedido", "devolucion"
embedding = model.encode(
    [frase_correcta_devolucion]
)
pred = clf.predict(embedding)
print(pred)
print (embedding)

['devolucion']
[[-4.62687649e-02  7.94534013e-02 -8.95635970e-03 -1.29857203e-02
  -8.81007984e-02 -6.15643896e-02  1.22304164e-01  1.05693601e-01
   2.64104139e-02  7.44826496e-02  8.34350586e-02 -8.07810426e-02
   2.16714712e-03  2.65408754e-02 -5.05653098e-02 -3.60051394e-02
  -4.68522981e-02  8.63063782e-02 -3.72972786e-02  4.46427688e-02
   7.49237090e-02 -4.59076315e-02 -8.82093757e-02  4.87305857e-02
  -3.37914005e-02  7.59177133e-02  1.31809246e-02  4.89416160e-02
   2.47624377e-03 -2.16833893e-02 -2.36197431e-02  8.91042501e-02
   1.06804699e-01 -8.99844393e-02  5.27508445e-02 -1.72228552e-02
   4.32206281e-02 -5.71660921e-02  4.62613143e-02 -5.94137050e-03
  -8.28386694e-02 -4.65200022e-02  1.46795074e-02 -1.35745471e-02
   7.56994402e-03 -1.75775774e-02  3.80827934e-02  8.74366537e-02
  -4.77560908e-02  1.16226841e-02 -4.13899720e-02 -1.44094024e-02
  -1.79813569e-03  4.12015952e-02  1.63377114e-02  1.20276131e-03
   5.92904203e-02  1.53078800e-02  1.46921808e-02  4.90747578

In [69]:
frase_correcta_devolucion = "quiero cambiar mi pedido", "devolucion"
embedding = model.encode(
    [frase_correcta_devolucion]
)
pred = clf.predict(embedding)
print(pred)
print(embedding)

['devolucion']
[[-4.32934947e-02  7.65491053e-02 -4.63553220e-02 -4.43851128e-02
  -6.49017319e-02 -5.53237274e-02  1.34175420e-01  1.16632596e-01
   5.06505445e-02  4.68773805e-02  9.50356200e-02 -1.03347912e-01
  -2.78987009e-02  4.87084612e-02 -2.89825890e-02 -5.37643544e-02
  -6.91459030e-02  5.88609055e-02 -3.22049670e-02  4.49705273e-02
  -2.51581408e-02 -5.38245179e-02 -5.80674186e-02  6.01032786e-02
  -3.53308469e-02  5.49985617e-02  1.85540915e-02  6.69667497e-02
  -9.85725876e-03 -2.51986720e-02 -2.74833222e-03  6.85186461e-02
   1.14427082e-01 -5.85119426e-02  3.94477062e-02 -3.34314406e-02
   3.59979756e-02 -4.21436131e-02  8.55731219e-03  6.75389171e-02
  -8.54061618e-02 -3.17294672e-02 -4.16224869e-03 -2.04739645e-02
   2.39206422e-02  1.13655347e-02  6.35353550e-02  8.62295777e-02
  -4.28759009e-02 -1.13795102e-02 -8.24481174e-02  3.84774455e-03
  -1.08240743e-03  1.51853850e-02 -1.30390916e-02  2.14667292e-03
   1.37335788e-02 -2.91667413e-03 -1.80953857e-03  4.19603176

In [70]:
frase_dudosa_saludo= "eguerdi on, dena ondo gazteok?"
embedding = model.encode(
    [frase_dudosa_saludo]
)
pred = clf.predict(embedding)
print(pred)
print(embedding)

['devolucion']
[[-1.13688342e-01  1.14627004e-01 -1.45866806e-02 -1.63656694e-03
  -6.04575872e-02 -1.39284367e-02  1.24705136e-01  1.04996309e-01
   1.08448761e-02  1.19127834e-03  2.87755635e-02 -4.42527570e-02
  -2.67344192e-02 -2.32302118e-02 -1.61193293e-02 -4.32360396e-02
  -3.35534699e-02  3.06476522e-02 -1.99648533e-02 -1.75151173e-02
  -2.15856507e-02 -3.39689106e-03 -2.22240537e-02  3.12612131e-02
  -2.72052027e-02 -8.03444013e-02  7.62260929e-02  1.90617926e-02
  -6.74517080e-02 -6.44298792e-02  1.23543860e-02 -4.48302142e-02
   4.49188314e-02  1.59385137e-03 -4.41989191e-02  5.90690114e-02
   2.16012243e-02 -7.39721879e-02  1.43454759e-03  7.64499456e-02
  -1.12021200e-01 -6.15232624e-03 -7.67137557e-02 -9.29428264e-02
  -3.53197195e-03 -5.17872386e-02 -8.73244107e-02  4.06604148e-02
  -3.73339765e-02 -2.96018198e-02 -7.23311082e-02 -2.60332115e-02
  -2.06804816e-02  4.38508838e-02  4.99064103e-02  6.82126954e-02
  -6.23179488e-02  2.98516694e-02  4.74757142e-02 -6.59447955

## Ejercicio 4 — Mejorar los datos de entrenamiento
El modelo del notebook utiliza muy pocos ejemplos. Investiga qué ocurre cuando aumentamos y diversificamos los datos.

Tareas:

11. Para cada una de las categorías crea al menos 5 ejemplos adicionales.

12. Intenta que los ejemplos de una misma categoría estén expresados de formas diferentes.

13. Incluye frases cortas y largas.

14. Incluye diferentes formas de expresar una misma intención.

15. Vuelve a generar los embeddings y entrena de nuevo el clasificador.

16. Repite las pruebas del ejercicio anterior y compara los resultados.
# Preguntas:
¿Mejora el comportamiento del modelo al añadir ejemplos?

Si, en este caso al tener mas ejemplo y varianza el modelo es capaz de predecir mejor

¿Qué ocurre si todos los ejemplos de una categoría son muy parecidos?

Que se produce un patron, por lo que es mas probable que el modelo solo asigne esa categoria si mantiene semejanza con el patron

¿Qué puede ocurrir si una categoría tiene muchos más ejemplos que las demás?

Que por probabilidad es mas probable que esa categoria salga mas como respuesta

¿Por qué la cantidad y variedad de los datos puede ser importante aunque utilicemos un modelo preentrenado?

Porque la informacion de estos modelos es muy general y debe adaptarse

Entrega: Comparación entre el clasificador original y el clasificador con más datos.

In [71]:
model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

embedding = model.encode(
    "quiero devolver mi pedido"
)

print(embedding.shape)

print(embedding[:10])

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3384.70it/s]


(384,)
[-0.01777516  0.06185898  0.00256953 -0.03221566 -0.07757559 -0.06224593
  0.09503879  0.05418092  0.05467332  0.06182623]


In [74]:
texts = [x[0] for x in train_data_mejorado]
labels = [x[1] for x in train_data_mejorado]

X = model.encode(texts)

In [75]:
# Entrenar el modelo
#clf = LogisticRegression()
# C10 permite al modelo ajustarse mejor a los datos de entrenamiento
# max_iter=1000 permite al modelo converger en más iteraciones si es necesario
clf = LogisticRegression(C=10, max_iter=1000) 

clf.fit(X, labels)

,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",10
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in

In [ ]:
frase_correcta_devolucion = "quiero modificar mi pedido", "devolucion"
embedding = model.encode(
    [frase_correcta_devolucion]
)
pred = clf.predict(embedding)
print(pred)
print(embedding)
embedding.shape



['devolucion']
[[-4.62687649e-02  7.94534013e-02 -8.95635970e-03 -1.29857203e-02
  -8.81007984e-02 -6.15643896e-02  1.22304164e-01  1.05693601e-01
   2.64104139e-02  7.44826496e-02  8.34350586e-02 -8.07810426e-02
   2.16714712e-03  2.65408754e-02 -5.05653098e-02 -3.60051394e-02
  -4.68522981e-02  8.63063782e-02 -3.72972786e-02  4.46427688e-02
   7.49237090e-02 -4.59076315e-02 -8.82093757e-02  4.87305857e-02
  -3.37914005e-02  7.59177133e-02  1.31809246e-02  4.89416160e-02
   2.47624377e-03 -2.16833893e-02 -2.36197431e-02  8.91042501e-02
   1.06804699e-01 -8.99844393e-02  5.27508445e-02 -1.72228552e-02
   4.32206281e-02 -5.71660921e-02  4.62613143e-02 -5.94137050e-03
  -8.28386694e-02 -4.65200022e-02  1.46795074e-02 -1.35745471e-02
   7.56994402e-03 -1.75775774e-02  3.80827934e-02  8.74366537e-02
  -4.77560908e-02  1.16226841e-02 -4.13899720e-02 -1.44094024e-02
  -1.79813569e-03  4.12015952e-02  1.63377114e-02  1.20276131e-03
   5.92904203e-02  1.53078800e-02  1.46921808e-02  4.90747578

In [80]:
frase_correcta_devolucion = "quiero cambiar mi pedido", "devolucion"
embedding = model.encode(
    [frase_correcta_devolucion]
)
pred = clf.predict(embedding)
print(pred)
print(embedding)

['devolucion']
[[-4.32934947e-02  7.65491053e-02 -4.63553220e-02 -4.43851128e-02
  -6.49017319e-02 -5.53237274e-02  1.34175420e-01  1.16632596e-01
   5.06505445e-02  4.68773805e-02  9.50356200e-02 -1.03347912e-01
  -2.78987009e-02  4.87084612e-02 -2.89825890e-02 -5.37643544e-02
  -6.91459030e-02  5.88609055e-02 -3.22049670e-02  4.49705273e-02
  -2.51581408e-02 -5.38245179e-02 -5.80674186e-02  6.01032786e-02
  -3.53308469e-02  5.49985617e-02  1.85540915e-02  6.69667497e-02
  -9.85725876e-03 -2.51986720e-02 -2.74833222e-03  6.85186461e-02
   1.14427082e-01 -5.85119426e-02  3.94477062e-02 -3.34314406e-02
   3.59979756e-02 -4.21436131e-02  8.55731219e-03  6.75389171e-02
  -8.54061618e-02 -3.17294672e-02 -4.16224869e-03 -2.04739645e-02
   2.39206422e-02  1.13655347e-02  6.35353550e-02  8.62295777e-02
  -4.28759009e-02 -1.13795102e-02 -8.24481174e-02  3.84774455e-03
  -1.08240743e-03  1.51853850e-02 -1.30390916e-02  2.14667292e-03
   1.37335788e-02 -2.91667413e-03 -1.80953857e-03  4.19603176

In [79]:
frase_dudosa_saludo= "eguerdi on, dena ondo gazteok?"
embedding = model.encode(
    [frase_dudosa_saludo]
)
pred = clf.predict(embedding)
print(pred)
print(embedding)
embedding.shape

['saludo']
[[-1.13688342e-01  1.14627004e-01 -1.45866806e-02 -1.63656694e-03
  -6.04575872e-02 -1.39284367e-02  1.24705136e-01  1.04996309e-01
   1.08448761e-02  1.19127834e-03  2.87755635e-02 -4.42527570e-02
  -2.67344192e-02 -2.32302118e-02 -1.61193293e-02 -4.32360396e-02
  -3.35534699e-02  3.06476522e-02 -1.99648533e-02 -1.75151173e-02
  -2.15856507e-02 -3.39689106e-03 -2.22240537e-02  3.12612131e-02
  -2.72052027e-02 -8.03444013e-02  7.62260929e-02  1.90617926e-02
  -6.74517080e-02 -6.44298792e-02  1.23543860e-02 -4.48302142e-02
   4.49188314e-02  1.59385137e-03 -4.41989191e-02  5.90690114e-02
   2.16012243e-02 -7.39721879e-02  1.43454759e-03  7.64499456e-02
  -1.12021200e-01 -6.15232624e-03 -7.67137557e-02 -9.29428264e-02
  -3.53197195e-03 -5.17872386e-02 -8.73244107e-02  4.06604148e-02
  -3.73339765e-02 -2.96018198e-02 -7.23311082e-02 -2.60332115e-02
  -2.06804816e-02  4.38508838e-02  4.99064103e-02  6.82126954e-02
  -6.23179488e-02  2.98516694e-02  4.74757142e-02 -6.59447955e-03

(1, 384)

In [81]:
frase_dudosa_problema= "se me ha roto el ordenador 5 años despues"
embedding = model.encode(
    [frase_dudosa_problema]
)
pred = clf.predict(embedding)
print(pred)
print (embedding)
embedding.shape

['problema']
[[-3.93888243e-02  2.12117992e-02  4.86572385e-02 -2.97098793e-02
   1.78603083e-02 -4.17333134e-02  5.58892414e-02  4.59684152e-03
   6.28773421e-02  6.29365295e-02  7.30500594e-02 -3.19025777e-02
   1.57768913e-02 -4.33597043e-02 -4.85576764e-02  2.95832232e-02
  -1.77120287e-02  7.86327794e-02 -9.07838196e-02  1.99212367e-03
   6.88346252e-02  1.66380703e-02 -1.33304432e-01  8.99142772e-02
  -1.10980824e-01  7.81193329e-03 -2.38560271e-02  1.48507664e-02
  -4.99075390e-02 -1.23652183e-01 -5.15041277e-02  1.90964490e-02
   7.28815794e-02  2.93730460e-02 -6.07938915e-02  2.02256674e-03
   6.20668121e-02 -8.02925453e-02 -6.10065311e-02  4.64914180e-02
  -6.99079558e-02  1.08474456e-02 -2.61404645e-02 -1.05031691e-02
   4.39610742e-02 -6.22576661e-02 -3.19065228e-02  2.24397294e-02
   6.42214268e-02 -2.79012993e-02 -2.01746095e-02  2.87292786e-02
   2.19115103e-03  1.68195814e-02 -7.19874576e-02  1.60182416e-02
   2.26415582e-02 -4.12965678e-02  9.05224010e-02 -5.73261641e-

(1, 384)

## Ejercicio 5 — Crear una función de clasificación
Convierte el código de predicción en una función que pueda utilizarse tantas veces como sea necesario.

Tareas:

17. Crea una función llamada clasificar_frase(frase).

18. La función debe recibir una frase como entrada.

19. Debe generar su embedding utilizando el modelo Sentence Transformer.

20. Debe utilizar el clasificador entrenado para obtener la categoría.

21. Debe devolver la categoría predicha.

22. Prueba la función con al menos 10 frases diferentes.

Preguntas:

¿Qué partes del proceso se repiten cada vez que queremos clasificar una frase?

¿Qué elementos deben entrenarse una sola vez?

¿Qué ventaja tiene encapsular la predicción dentro de una función?

Entrega: Código de la función y ejemplos de utilización.

In [89]:
def clasificar_frase(frase):
    embedding = model.encode([frase])
    pred = clf.predict(embedding)
    return pred[0]

# Pruebas de la función de clasificación

ejemplos = [
    "quiero comprar un ordenador",
    "necesito un portátil nuevo",
    "quiero cambiar mi pedido",
    "quiero un reembolso",
    "hola",
    "buenos días",
    "mi ordenador no funciona",
    "no puedo acceder a mi cuenta",
    "kaixo",
    "el paquete ha llegado defectuoso",
    "¿cuánto cuesta este teléfono?",
    "ayuda, mi pedido no ha llegado"
]

# Clasificar e imprimir los resultados
for frase in ejemplos:
    categoria = clasificar_frase(frase)
    print(f"Frase: {frase}")
    print(f"Categoría predicha: {categoria}")
    print("-" * 50)

Frase: quiero comprar un ordenador
Categoría predicha: compra
--------------------------------------------------
Frase: necesito un portátil nuevo
Categoría predicha: compra
--------------------------------------------------
Frase: quiero cambiar mi pedido
Categoría predicha: devolucion
--------------------------------------------------
Frase: quiero un reembolso
Categoría predicha: devolucion
--------------------------------------------------
Frase: hola
Categoría predicha: saludo
--------------------------------------------------
Frase: buenos días
Categoría predicha: saludo
--------------------------------------------------
Frase: mi ordenador no funciona
Categoría predicha: problema
--------------------------------------------------
Frase: no puedo acceder a mi cuenta
Categoría predicha: problema
--------------------------------------------------
Frase: kaixo
Categoría predicha: saludo
--------------------------------------------------
Frase: el paquete ha llegado defectuoso
Catego